# GitHub Ref-CAS 시뮬레이션, 참조를 어디에 두면 push 가 빨라지나

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/git_ref_cas.ipynb)

같은 수의 에이전트가 같은 평균 생각 시간으로 push할 때, 참조를 단일 blob, 분할 blob, 복제 로그(Ref-CAS)에 두면 초당 몇 건이 도착하고 확정되는지 비교합니다.
push가 도착해서 참조 갱신이 확정되기까지의 시간과, 현재 참조 상태를 읽는 시간을 따로 잽니다.
같은 브랜치에 쓰는 비율과 로그 복제본 수를 바꿔 재시도 횟수와 지연의 99백분위수(p99)를 살펴봅니다.

**모든 지연 분포는 가정값입니다.** 표와 그래프의 수치는 모형에서 생성되며, GitHub 운영 환경의 실측 성능을 나타내지 않습니다. [GitHub 원문 · Inside GitHub's new Git storage architecture (2026-11-05)](https://github.blog/engineering/architecture-optimization/inside-githubs-new-git-storage-architecture/)

## 실행 설정

위에서부터 순서대로 실행하세요. 기본값은 시드 3개로 각 조건을 10초씩 모의하고 처음 1초를 워밍업으로 사용합니다. `QUICK = True`는 부하를 세 단계로 줄이고 시드 하나로 2초 중 처음 0.5초를 워밍업에 씁니다. 그래프 언어는 `ko`, `en`, `both` 중에서 선택합니다.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timezone
import heapq
import json
import math
import os
from pathlib import Path
import platform
import sys
import time
import unicodedata
from urllib.error import URLError
from urllib.request import urlopen

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

QUICK = False
CHART_LANG = os.environ.get("CHART_LANG", "ko")

BLOB_RTT_MS = 8
BLOB_SIGMA = 0.35
BATCH_MAX = 8
PER_UPDATE_MS = 0.4
SEGMENTS = (1, 4, 16, 64)
REPLICAS = (3, 5)
LOG_APPEND_MS = 2
LOG_SIGMA = 0.3
LEADER_MS = 0.05
PIPELINE = 256
VERSION_CHECK_MS = 1
VERSION_SIGMA = 0.3
RETRY_BACKOFF_MS = 1
MAX_RETRIES = 10
AGENTS = 5000
THINK_MS = 1000 / 1.44
BASE_LOAD = round(AGENTS * 1000 / THINK_MS)
LOADS = (500, 1000, 2000, 4000, BASE_LOAD, 14400)
QUICK_LOADS = (500, BASE_LOAD, 14400)
SHARED = (0, 0.01, 0.05, 0.2, 1.0)
RUN_MS, WARMUP_MS, SEEDS = 10000, 1000, (0, 1, 2)
QUICK_MS, QUICK_WARMUP_MS, QUICK_SEEDS = 2000, 500, (0,)
READ_SAMPLES = 20000

end, warmup, seeds, loads = RUN_MS, WARMUP_MS, SEEDS, LOADS
if QUICK:
    end, warmup = QUICK_MS, QUICK_WARMUP_MS
    seeds, loads = QUICK_SEEDS, QUICK_LOADS

if CHART_LANG not in ("ko", "en", "both"):
    raise ValueError("CHART_LANG must be ko, en, or both")
languages = (CHART_LANG,)
if CHART_LANG == "both":
    languages = ("ko", "en")
RESULTS = Path("results")

In [ ]:
stamp = datetime.now(timezone.utc).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)


def width(text):
    return sum(
        2 if unicodedata.east_asian_width(c) in "WF" else 1
        for c in text
    )


def line(text=""):
    if width(text) > 64:
        raise ValueError(f"Output exceeds 64 columns: {text}")
    print(text)


def header(stamp, quick, lang):
    for s in (
        f"실행 UTC {stamp}   {'quick' if quick else 'full'}",
        f"Python {platform.python_version()}   "
        f"numpy {np.__version__}   CHART_LANG={lang}",
        "모든 지연 분포는 시뮬레이션 가정이며 "
        "GitHub 실측이 아닙니다.",
        "로그정규 표기: 중앙값(ms), 자연로그 표준편차 σ",
        f"BLOB_RTT_MS={BLOB_RTT_MS}  σ={BLOB_SIGMA}"
        " (조건부 쓰기·상태 조회)",
        f"BATCH_MAX={BATCH_MAX}  PER_UPDATE_MS={PER_UPDATE_MS}"
        " (push 1건당)",
        f"K={SEGMENTS}  읽기=max(K회 병렬 조회)",
        f"R={REPLICAS}  LOG_APPEND_MS={LOG_APPEND_MS}  "
        f"σ={LOG_SIGMA}",
        f"LEADER_MS={LEADER_MS}  PIPELINE={PIPELINE}  "
        "확정=과반째 기록+리더",
        f"VERSION_CHECK_MS={VERSION_CHECK_MS}  "
        f"σ={VERSION_SIGMA} (최신 LSN 비교)",
        f"RETRY_BACKOFF_MS={RETRY_BACKOFF_MS}  최대 "
        f"{MAX_RETRIES}회  대기=시작값×2^실패인덱스",
        "실패 후 대기·fetch 1회, fetch 완료 시 기대 버전 취득",
        f"N={AGENTS}  THINK_MS={THINK_MS:.3f} "
        "(기본 1.44회/초·명)",
        "에이전트마다 지수분포 생각 → "
        "push 확정·포기 → 다시 생각",
        "목표 속도=N×1000/THINK_MS, "
        "실제 도착률은 지연만큼 낮아짐",
        f"전체 목표(/s)={','.join(map(str, LOADS))}",
        f"quick 목표(/s)={','.join(map(str, QUICK_LOADS))}",
        f"shared_frac={','.join(map(str, SHARED))} "
        "(공유 참조 하나)",
        "K=blob 분할 수(1은 단일), R=Ref-CAS 복제본 수",
        f"실험 1·2·4 공유 0%, 실험 2·3·4 목표 {BASE_LOAD:,}/s",
        f"전체 {RUN_MS}ms / 워밍업 {WARMUP_MS}ms / "
        f"시드 {SEEDS}",
        f"quick {QUICK_MS}ms / 워밍업 {QUICK_WARMUP_MS}ms / "
        f"시드 {QUICK_SEEDS}",
        f"읽기 표본={READ_SAMPLES}/시드, "
        "독립 RTT, 전송량 비용 생략",
        "대기열 FIFO, 최초 도착 시 "
        "현재 참조 버전을 기대값으로 사용",
        "도착/s는 구간 내 새 push, 시도/s는 재시도 포함",
        "처리량·지연은 관측 구간 내 확정 기준, "
        "지연은 최초 도착부터",
        "실패/push·포기율 분모는 구간 내 확정 또는 포기한 push",
        "표는 시드별 지표의 중앙값·범위 (quick은 단일 시드)",
    ):
        line(s)


header(stamp, QUICK, CHART_LANG)

## 그래프 글꼴

한국어 그래프를 그릴 때는 NanumGothic을 `cache/fonts/`에 한 번 내려받아 Matplotlib에 등록합니다. 다운로드할 수 없으면 설치된 한글 글꼴을 찾습니다. [Google Fonts · NanumGothic, SIL Open Font License 1.1](https://github.com/google/fonts/tree/main/ofl/nanumgothic)

In [ ]:
FONT_URL = (
    "https://github.com/google/fonts/raw/main/"
    "ofl/nanumgothic/NanumGothic-Regular.ttf"
)


def configure_font():
    font = Path("cache/fonts/NanumGothic-Regular.ttf")
    try:
        if not font.exists():
            with urlopen(FONT_URL, timeout=5) as response:
                contents = response.read()
            font.parent.mkdir(parents=True, exist_ok=True)
            font.write_bytes(contents)
        font_manager.fontManager.addfont(font)
        props = font_manager.FontProperties(fname=font)
        plt.rcParams["font.family"] = props.get_name()
    except (OSError, RuntimeError, URLError) as error:
        print(
            "경고: NanumGothic 사용 불가 "
            f"({type(error).__name__}). 시스템 기본 글꼴 사용.",
            file=sys.stderr,
        )
        names = font_manager.fontManager.get_font_names()
        for family in (
            "Apple SD Gothic Neo", "Malgun Gothic",
            "Noto Sans CJK KR",
        ):
            if family in names:
                fonts = plt.rcParams["font.sans-serif"]
                fonts.insert(0, family)
                break


if "ko" in languages:
    configure_font()
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False, "font.size": 10,
})

## 지연 분포

왕복 시간(RTT)은 로그정규분포에서 뽑습니다. 설정의 시간 단위는 밀리초이고 σ는 자연로그의 표준편차입니다. 병렬 요청은 같은 행에서 여러 번 뽑아, 완료를 기다려야 하는 순번의 값 하나를 사용합니다.

In [ ]:
class Delays:
    def __init__(self, rng, median, sigma, count=1, order=0):
        self.rng = rng
        self.mu = math.log(median)
        self.sigma = sigma
        self.count, self.order = count, order
        self.values = iter(())

    def next(self):
        value = next(self.values, None)
        if value is None:
            samples = self.rng.lognormal(
                self.mu, self.sigma, (2048, self.count)
            )
            if self.count > 1:
                samples.partition(self.order, axis=1)
            self.values = iter(samples[:, self.order].tolist())
            value = next(self.values)
        return value


def read_metrics(seed):
    rng = np.random.default_rng([seed, 92002])
    samples = rng.lognormal(
        math.log(BLOB_RTT_MS), BLOB_SIGMA,
        (READ_SAMPLES, max(SEGMENTS)),
    )
    times = {
        ("blob", k): samples[:, :k].max(axis=1)
        for k in SEGMENTS
    }
    times["log", 3] = rng.lognormal(
        math.log(VERSION_CHECK_MS), VERSION_SIGMA, READ_SAMPLES
    )
    fields = ("read_p50_ms", "read_p99_ms")
    return {
        key: dict(zip(fields, np.percentile(values, [50, 99])))
        for key, values in times.items()
    }

## 사건 큐와 관측 구간

큐는 사건을 시간순으로 처리하고, 시각이 같으면 먼저 넣은 사건부터 꺼냅니다. push는 도착할 때의 참조 버전을 기억합니다. 비교에 실패하면 지수 백오프와 읽기 한 번을 거친 뒤 그때의 버전으로 재시도합니다.

워밍업 중에도 대기열과 버전은 계속 갱신합니다. 처리량과 지연은 관측 구간에 확정된 push로 계산하고, 실패 횟수와 포기율은 그 구간에 확정되거나 포기한 push로 계산합니다. 종료 시 대기 중인 건수는 JSON의 `pending_at_end`에 남습니다.

### 단일 blob과 분할 blob

참조를 해시해 K개 분할 중 하나의 FIFO 대기열에 넣습니다. K가 1이면 모두 같은 큐에 들어갑니다. 한 번에 최대 8건을 묶어 기록하고, 완료 시점에 묶음 안의 순서대로 기대 버전을 검사합니다. 모든 참조를 읽을 때는 K개 병렬 조회 중 마지막 응답을 기다립니다.

### Ref-CAS와 복제 로그

Ref-CAS는 갱신할 참조의 기대 버전이 현재 버전과 일치하는지 검사합니다. 일치하면 그 참조를 잠그고 복제를 시작합니다. 서로 다른 참조는 최대 256개까지 동시에 기록합니다. 과반의 복제본이 응답하면 리더 처리 시간을 더해 확정하고, 새 로그 순번(LSN)을 해당 참조의 버전으로 기록합니다.

In [ ]:
@dataclass(slots=True)
class Push:
    start: float
    agent: int
    ref: int
    expected: int
    failures: int = 0


@dataclass(frozen=True)
class Case:
    mode: str
    size: int
    rate: int
    shared: float = 0

    @property
    def think_ms(self):
        return AGENTS * 1000 / self.rate


# Queue entries: (time, sequence, event kind, payload).
# Replica writes and parallel reads use one order statistic.
ARRIVAL, RETRY, BLOB_DONE, LOG_DONE = range(4)


class Simulation:
    def __init__(self, case, end, warmup, seed):
        self.case, self.end, self.warmup = case, end, warmup
        self.think_rng = np.random.default_rng(
            [seed, case.rate]
        )
        self.window_arrivals = self.window_retries = 0
        self.arrived = self.sequence = self.events = 0
        self.lsn = 0
        self.heap = []
        self.version = [0] * (AGENTS + 1)
        self.latencies = []
        self.gave_up = self.terminal_failures = 0
        self.failed_checks = 0
        self.terminal_all = 0
        self.shared_agents = round(AGENTS * case.shared)
        self.is_log = case.mode == "log"
        service_rng, fetch_rng = [
            np.random.default_rng(s) for s in
            np.random.SeedSequence([seed, 92001]).spawn(2)
        ]
        if self.is_log:
            self.service = Delays(
                service_rng, LOG_APPEND_MS, LOG_SIGMA,
                case.size, case.size // 2,
            )
            self.fetch = Delays(
                fetch_rng, VERSION_CHECK_MS, VERSION_SIGMA
            )
            self.locked = set()
            self.waiting = defaultdict(deque)
            # One entry per runnable ref, oldest first.
            self.ready = deque()
        else:
            self.service = Delays(
                service_rng, BLOB_RTT_MS, BLOB_SIGMA
            )
            self.fetch = Delays(
                fetch_rng, BLOB_RTT_MS, BLOB_SIGMA,
                case.size, case.size - 1,
            )
            self.segments = [deque() for _ in range(case.size)]
            self.busy = [False] * case.size

    def schedule(self, when, kind, payload):
        self.sequence += 1
        heapq.heappush(
            self.heap, (when, self.sequence, kind, payload)
        )

    def failed(self, now, push):
        push.failures += 1
        self.failed_checks += int(now >= self.warmup)
        if push.failures > MAX_RETRIES:
            self.finish(now, push, gave_up=True)
            return
        backoff = RETRY_BACKOFF_MS * 2 ** (push.failures - 1)
        self.schedule(
            now + backoff + self.fetch.next(), RETRY, push
        )

    def finish(self, now, push, gave_up=False):
        self.terminal_all += 1
        if now >= self.warmup:
            if gave_up:
                self.gave_up += 1
            else:
                self.latencies.append(now - push.start)
            self.terminal_failures += push.failures
        self.schedule(
            now + self.think_rng.exponential(
                self.case.think_ms
            ),
            ARRIVAL, push.agent,
        )

    def start_batch(self, now, segment):
        queue = self.segments[segment]
        self.busy[segment] = bool(queue)
        if queue:
            batch = [
                queue.popleft()
                for _ in range(min(len(queue), BATCH_MAX))
            ]
            delay = (
                self.service.next()
                + PER_UPDATE_MS * len(batch)
            )
            self.schedule(
                now + delay, BLOB_DONE, (segment, batch)
            )

    def dispatch_log(self, now):
        while self.ready and len(self.locked) < PIPELINE:
            ref = self.ready.popleft()
            queue = self.waiting[ref]
            # A stale comparison releases the lock immediately.
            # Only one push per version can enter the log.
            while queue:
                push = queue.popleft()
                if push.expected == self.version[ref]:
                    self.locked.add(ref)
                    self.schedule(
                        now + LEADER_MS + self.service.next(),
                        LOG_DONE, push,
                    )
                    break
                self.failed(now, push)
            if not queue:
                del self.waiting[ref]

    def submit(self, now, push):
        if self.is_log:
            queue = self.waiting[push.ref]
            if not queue and push.ref not in self.locked:
                self.ready.append(push.ref)
            queue.append(push)
            self.dispatch_log(now)
        else:
            segment = (
                ((push.ref * 2654435761) & 0xffffffff)
                % self.case.size
            )
            self.segments[segment].append(push)
            if not self.busy[segment]:
                self.start_batch(now, segment)

    def run(self):
        for agent, delay in enumerate(
            self.think_rng.exponential(
                self.case.think_ms, AGENTS
            )
        ):
            self.schedule(delay, ARRIVAL, agent)
        while self.heap:
            now, _, kind, data = heapq.heappop(self.heap)
            if now >= self.end:
                break
            self.events += 1
            if kind == ARRIVAL:
                agent = data
                ref = (
                    0 if agent < self.shared_agents
                    else agent + 1
                )
                self.submit(
                    now,
                    Push(now, agent, ref, self.version[ref]),
                )
                self.arrived += 1
                self.window_arrivals += int(now >= self.warmup)
            elif kind == RETRY:
                self.window_retries += int(now >= self.warmup)
                data.expected = self.version[data.ref]
                self.submit(now, data)
            elif kind == LOG_DONE:
                self.lsn += 1
                self.version[data.ref] = self.lsn
                self.finish(now, data)
                self.locked.remove(data.ref)
                if data.ref in self.waiting:
                    self.ready.append(data.ref)
                self.dispatch_log(now)
            else:
                segment, batch = data
                for push in batch:
                    if push.expected == self.version[push.ref]:
                        self.version[push.ref] += 1
                        self.finish(now, push)
                    else:
                        self.failed(now, push)
                self.start_batch(now, segment)
        count = len(self.latencies)
        terminal = count + self.gave_up
        seconds = (self.end - self.warmup) / 1000
        p50, p99 = (
            np.percentile(self.latencies, [50, 99])
            if count else (None, None)
        )
        return dict(
            arrival_rate_s=self.window_arrivals / seconds,
            attempt_rate_s=(
                self.window_arrivals + self.window_retries
            ) / seconds,
            throughput_s=count / seconds,
            commit_p50_ms=p50, commit_p99_ms=p99,
            failures_per_push=(
                self.terminal_failures / terminal
                if terminal else None
            ),
            giveup_fraction=(
                self.gave_up / terminal if terminal else None
            ),
            committed=count, gave_up=self.gave_up,
            terminal_push_failures=self.terminal_failures,
            cas_failures_in_window=self.failed_checks,
            arrivals_all=self.arrived,
            committed_or_gave_up_all=self.terminal_all,
            pending_at_end=self.arrived - self.terminal_all,
            latest_lsn=self.lsn if self.is_log else None,
            events_all=self.events,
        )

## 부하 생성과 실험 조건

5,000명의 에이전트가 각자 지수분포에서 뽑은 시간만큼 생각한 뒤 push합니다. 확정되거나 재시도 한도에 도달해 포기할 때까지 기다린 뒤 다시 생각합니다. 공유 그룹은 하나의 참조에 쓰고 나머지는 각자 자기 참조에 씁니다.

기본 목표 속도는 `N × 1000 / THINK_MS`입니다. 실험 1의 각 목표 속도에 맞춰 평균 생각 시간을 다시 계산합니다. 같은 시드와 목표 속도에서는 에이전트별 첫 생각 시간이 같습니다. 이후 도착 시각은 push가 끝나는 시각에 따라 달라집니다. 여러 실험에 겹치는 조건은 시드마다 한 번만 계산합니다.

In [ ]:
def experiments(loads):
    stores = (("blob", 1), ("blob", 16), ("log", 3))
    return {
        "throughput_vs_load": [
            Case(mode, size, rate)
            for rate in loads for mode, size in stores
        ],
        "read_latency_by_segments": [
            Case("blob", k, BASE_LOAD) for k in SEGMENTS
        ] + [Case("log", 3, BASE_LOAD)],
        "shared_branch": [
            Case("log", 3, BASE_LOAD, f) for f in SHARED
        ],
        "replicas": [
            Case("log", r, BASE_LOAD) for r in REPLICAS
        ],
    }


design = experiments(loads)
unique = dict.fromkeys(
    c for cases in design.values() for c in cases
)
raw = {c: [] for c in unique}
measured = {c: dict(events=0, seconds=0.0) for c in unique}
for seed in seeds:
    reads = read_metrics(seed)
    for case in unique:
        start = time.perf_counter()
        row = Simulation(case, end, warmup, seed).run()
        measured[case]["seconds"] += time.perf_counter() - start
        measured[case]["events"] += row["events_all"]
        raw[case].append(dict(
            seed=seed, **row,
            **reads.get((case.mode, case.size), {}),
        ))
data = {
    key: [dict(
        mode=c.mode, size=c.size, rate=c.rate,
        think_ms=c.think_ms, shared=c.shared, seeds=raw[c],
    ) for c in cases]
    for key, cases in design.items()
}

## 표와 그래프 읽기

표의 중앙·최소·최대는 각 시드에서 구한 지표를 시드 간에 요약한 값입니다. 축소 실행에서는 시드가 하나여서 중앙값만 표시합니다. 그래프는 중앙값을 사용하며, 실행하면 셀에 표시되고 같은 그림이 `results/`에 PNG로 저장됩니다.

In [ ]:
def table(rows):
    widths = [max(map(width, col)) for col in zip(*rows)]
    for row in rows:
        line("  ".join(
            " " * (w - width(cell)) + cell
            for w, cell in zip(widths, row)
        ))


ARR = ("arrival_rate_s", "도착/s", 1, 1)
TPS = ("throughput_s", "확정/s", 1, 1)
P50 = ("commit_p50_ms", "p50 ms", 2, 1)
P99 = ("commit_p99_ms", "p99 ms", 2, 1)


def report(key, heading, metrics):
    line()
    line(heading)
    rows = [["조건", "통계"] + [m[1] for m in metrics]]
    for entry in data[key]:
        samples = entry["seeds"]
        size = entry["size"]
        name = (
            f"K={size}" if entry["mode"] == "blob"
            else f"R={size}"
        )
        if key == "throughput_vs_load":
            name = f"{entry['rate']}/s {name}"
        elif key == "shared_branch":
            name = f"{entry['shared'] * 100:g}%"
        stats = [("중앙", np.median)]
        if len(samples) > 1:
            stats += [("최소", min), ("최대", max)]
        for i, (label, stat) in enumerate(stats):
            row = [name if i == 0 else "", label]
            for field, _, places, factor in metrics:
                values = [s[field] for s in samples
                          if s[field] is not None]
                cell = "—"
                if values:
                    cell = f"{stat(values) * factor:.{places}f}"
                row.append(cell)
            rows.append(row)
    table(rows)


def med(entry, key):
    return np.median([s[key] for s in entry["seeds"]])


colors = ("#527796", "#518675", "#96674f")
RESULTS.mkdir(exist_ok=True)


def axes(xlabel, ylabel):
    fig, ax = plt.subplots(
        figsize=(6.2, 3.8), layout="constrained"
    )
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.18)
    return fig, ax


def save(fig, name, language):
    suffix = f"_{language}" if CHART_LANG == "both" else ""
    fig.savefig(RESULTS / f"{name}{suffix}.png", dpi=150)
    plt.show()
    plt.close(fig)

## 실험 1 · 목표 속도와 실제 도착률·확정 처리량

각 에이전트가 자기 브랜치에 push할 때 평균 생각 시간을 줄여 목표 속도를 높입니다. 점선은 목표 속도만큼 확정할 때의 처리량입니다. 표의 `도착/s`는 push 확정이나 포기를 기다리는 시간까지 반영한 실제 도착률입니다. 곡선이 점선 아래로 벌어지는 구간에서 도착률과 확정 지연의 p50·p99를 함께 보세요.

In [ ]:
report(
    "throughput_vs_load",
    "1. 목표 속도와 실제 도착률·확정 처리량·커밋 지연",
    (ARR, TPS, P50, P99),
)
exp1 = data["throughput_vs_load"]
target_rates = sorted({e["rate"] for e in exp1})
for language in languages:
    ko = language == "ko"
    fig, ax = axes(
        "목표 속도 (push/s)" if ko else "Target rate (push/s)",
        "확정 처리량 (push/s)" if ko
        else "Committed throughput (push/s)",
    )
    ax.loglog(
        target_rates, target_rates, "--",
        color="0.65", lw=1, zorder=1,
    )
    series = (
        ("blob", 1, "단일 blob" if ko else "Single blob", "o"),
        ("blob", 16, "분할 blob (K=16)" if ko
         else "Segmented blob (K=16)", "s"),
        ("log", 3, "Ref-CAS (R=3)", "^"),
    )
    for config, color in zip(series, colors):
        mode, size, label, marker = config
        values = [
            med(e, "throughput_s") for e in exp1
            if (e["mode"], e["size"]) == (mode, size)
        ]
        ax.plot(
            target_rates, values, marker=marker, color=color,
            label=label, markersize=8 if size == 16 else 5,
            markerfacecolor="none" if size == 16 else color,
        )
    ax.legend(frameon=False)
    save(fig, "fig1_throughput_vs_load", language)

## 실험 2 · 분할 수와 읽기 지연

목표 속도를 7,200 push/s로 고정하고, blob을 1·4·16·64개로 나눕니다. 막대는 모든 분할의 응답이 올 때까지 걸리는 시간의 p99입니다. 수평선은 최신 LSN을 가진 클라이언트가 Ref-CAS에 버전을 한 번 확인하는 경우입니다.

In [ ]:
report(
    "read_latency_by_segments",
    "2. 목표 7,200/s에서의 확정 처리량·읽기 지연",
    (TPS, ("read_p50_ms", "읽기50 ms", 2, 1),
     ("read_p99_ms", "읽기99 ms", 2, 1)),
)
exp2 = data["read_latency_by_segments"]
for language in languages:
    ko = language == "ko"
    fig, ax = axes(
        "분할 수 (K)" if ko else "Segment count (K)",
        "읽기 지연 p99 (ms)" if ko else "p99 read latency (ms)",
    )
    blobs = [e for e in exp2 if e["mode"] == "blob"]
    ax.bar(
        [str(e["size"]) for e in blobs],
        [med(e, "read_p99_ms") for e in blobs],
        color=colors[0], width=0.58, label="Blob",
    )
    ax.axhline(
        med(exp2[-1], "read_p99_ms"), color=colors[2],
        ls="--", label="Ref-CAS",
    )
    ax.legend(frameon=False)
    save(fig, "fig2_read_latency_by_segments", language)

## 실험 3 · 공유 브랜치와 CAS 재시도

Ref-CAS 복제본 3개와 목표 속도 7,200 push/s에서 같은 브랜치에 쓰는 에이전트 비율을 0%부터 100%까지 높입니다. 그래프는 확정되거나 포기한 push 한 건당 평균 CAS 실패 횟수입니다. 표에는 새 push의 도착률, 재시도를 포함한 `시도/s`, 재시도 한도에 도달해 포기한 비율을 함께 표시합니다.

In [ ]:
report(
    "shared_branch", "3. 공유 브랜치 비율과 CAS 경합",
    (
        ARR, ("attempt_rate_s", "시도/s", 1, 1), TPS, P99,
        ("failures_per_push", "실패/push", 4, 1),
        ("giveup_fraction", "포기 %", 3, 100),
    ),
)
exp3 = data["shared_branch"]
for language in languages:
    ko = language == "ko"
    fig, ax = axes(
        "공유 브랜치 에이전트 비율" if ko
        else "Agents on shared branch",
        "push당 평균 CAS 실패 횟수" if ko
        else "Mean CAS failures per push",
    )
    ax.bar(
        [f"{e['shared'] * 100:g}%" for e in exp3],
        [med(e, "failures_per_push") for e in exp3],
        color=colors[1], width=0.58,
    )
    save(fig, "fig3_shared_branch", language)

## 실험 4 · 복제본 수와 확정 지연

각 에이전트가 자기 브랜치에 쓰는 조건에서 Ref-CAS 복제본을 3개에서 5개로 늘립니다. 목표 속도 7,200 push/s에서 확정에 필요한 응답이 두 번째에서 세 번째로 바뀔 때의 p50과 p99를 비교합니다.

In [ ]:
report(
    "replicas", "4. 복제본 수와 커밋 지연", (P50, P99),
)

## 시드별 결과 저장과 실행 시간

`results/results.json`에 네 실험의 조건(평균 생각 시간 포함), 시드별 지표, 실행 환경을 저장합니다. 축소 실행에서 전체 시간을 추정할 때는 같은 저장 방식·크기·공유 비율의 목표 속도별 측정값을 보간합니다.

모형은 독립적인 왕복 시간을 가정하며 네트워크 대역폭, 장애, Git 객체와 변경된 참조 목록의 전송 비용을 생략합니다. 큐가 포화되면 확정 지연은 관측 구간의 길이에도 영향을 받습니다.

In [ ]:
def full_estimate(measured, end, nseeds):
    events = seconds = 0.0
    groups = experiments(LOADS).values()
    cases = {case for group in groups for case in group}
    for case in cases:
        matches = sorted(
            (c.rate, record) for c, record in measured.items()
            if (c.mode, c.size, c.shared)
            == (case.mode, case.size, case.shared)
        )
        rates = [rate for rate, _ in matches]
        scale = RUN_MS * len(SEEDS) * case.rate / (end * nseeds)
        for key in ("events", "seconds"):
            per_push = [r[key] / rate for rate, r in matches]
            rate = case.rate
            value = scale * np.interp(rate, rates, per_push)
            if key == "events":
                events += value
            else:
                seconds += value
    return round(events), seconds


est_events, est_s = full_estimate(measured, end, len(seeds))
events = sum(r["events"] for r in measured.values())
line()
line(f"실제 큐 사건 {events:,}개 (중복 조건 재사용)")
line(f"전체 추정 {est_events:,}개 / "
     f"이 CPU 시뮬레이션 {est_s:.1f}초")

metadata = dict(
    utc=stamp, quick=QUICK, python=platform.python_version(),
    numpy=np.__version__, duration_ms=end, warmup_ms=warmup,
    seeds=seeds, full_estimated_events=est_events,
    full_estimated_sim_seconds=est_s,
)
result_path = RESULTS / "results.json"
with result_path.open("w", encoding="utf-8") as f:
    json.dump(dict(metadata=metadata, experiments=data), f,
              indent=2, allow_nan=False)
    f.write("\n")